# 04B - Overfitting And Tree Diagnostics

Complementary audit of the operational top model defined in notebook 04. It does not train or score new sources.


In [ ]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Force an interactive notebook backend when running cells manually.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass
from IPython.display import Markdown, Image, display

try:
    import seaborn as sns
except ImportError:
    sns = None

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "font.size": 10,
})

PALETTE = {
    "xgboost": "#2F6F9F",
    "random_forest": "#3A9D5D",
    "hist_gradient_boosting": "#B7791F",
    "logistic_regression": "#7B61A8",
    "wr": "#A8324A",
    "candidate": "#2F6F9F",
    "negative": "#6B7280",
    "muted": "#6B7280",
}

def require_path(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def pct(value):
    return "-" if pd.isna(value) else f"{100 * float(value):.1f}%"

def model_display(row):
    variant = str(row.get("dataset_variant", ""))
    family = "Relaxed" if variant.startswith("relaxed") else "Strict"
    dataset = variant.replace("relaxed_", "").replace("strict_", "")
    dataset = dataset.replace("photometry", "Phot").replace("parallax_soft", "Parallax+").replace("poe_", "POE>=")
    feat = "+err" if str(row.get("feature_set", "")).endswith("error") else "base"
    model = {"xgboost": "XGB", "random_forest": "RF", "hist_gradient_boosting": "HGB", "logistic_regression": "LogReg"}.get(str(row.get("model", "")), str(row.get("model", "")))
    sampler = {"smote": "SMOTE", "smote_enn": "SMOTE-ENN", "none": "no sampler"}.get(str(row.get("sampler", "")), str(row.get("sampler", "")))
    return f"{model} {sampler} | {family} {dataset} | {feat}"

def add_model_columns(df):
    out = df.copy()
    out["model_label"] = out.apply(model_display, axis=1)
    for k in [50, 100, 500, 1000]:
        c = f"holdout_wr_at_{k}"
        if c in out.columns:
            out[f"{c}_pct"] = out[c] / out["wr_holdout"].replace(0, np.nan)
    return out

def rank_models(df, k=100):
    return df.sort_values(
        [f"holdout_wr_at_{k}_pct", "holdout_average_precision", "holdout_recall_at_100", "holdout_f2_wr"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

results = add_model_columns(results)
ranked = rank_models(results, k=100)
print(f"Configuraciones evaluadas: {len(results)}")
print(f"Aceptadas: {int(results['selection_status'].eq('accepted').sum())}")
print("Holdout WR by variant:")
display(results.drop_duplicates("dataset_variant").set_index("dataset_variant")["wr_holdout"].sort_index())


## Run Selection

This section lists runs synchronized in `training_history.duckdb`. Set `TRAINING_RUN_ID` to one available ID to inspect a specific training run.

In [ ]:
from wr_detector.modeling.notebook_runs import available_training_runs, load_training_results_for_notebook

available_runs = available_training_runs(ROOT)
display(available_runs)

TRAINING_RUN_ID = "run_20260608T194455984398Z_d1dee0800bd4"
results = load_training_results_for_notebook(ROOT, TRAINING_RUN_ID)
display(Markdown(f"**Active run:** `{TRAINING_RUN_ID}` with `{len(results)}` results."))

In [ ]:

HISTORY_DB = ROOT / "data/databases/training_history.duckdb"
WR_DB = ROOT / "data/databases/wr_reference.duckdb"


def wr_family(value):
    text = "" if pd.isna(value) else str(value).upper().strip()
    if "WN" in text and "WC" in text:
        return "WN/WC"
    if "WO" in text:
        return "WO"
    if "WC" in text:
        return "WC"
    if "WN" in text:
        return "WN"
    return "other/unknown" if text else "unknown"


def load_wr_catalog():
    if not WR_DB.exists():
        return pd.DataFrame(columns=["source_id", "wr_id", "spectral_type", "wr_family"])
    con = duckdb.connect(str(WR_DB), read_only=True)
    try:
        wr = con.execute('SELECT source_id, wr_id, "Spectral Type" AS spectral_type FROM wr_reference').fetchdf()
    finally:
        con.close()
    wr["wr_family"] = wr["spectral_type"].map(wr_family)
    return wr


WR_CATALOG = load_wr_catalog()


def load_predictions_for_row(row):
    if HISTORY_DB.exists():
        con = duckdb.connect(str(HISTORY_DB), read_only=True)
        try:
            cols = {r[1] for r in con.execute("PRAGMA table_info('model_predictions')").fetchall()}
            if {"source_id", "row_id"}.issubset(cols):
                pred = con.execute('''
                    SELECT split, row_id, source_id, target, score, predicted, threshold
                    FROM model_predictions
                    WHERE run_id = ? AND dataset_variant = ? AND feature_set = ? AND model = ? AND sampler = ?
                ''', [TRAINING_RUN_ID, row.dataset_variant, row.feature_set, row.model, row.sampler]).fetchdf()
                if not pred.empty:
                    return pred
        finally:
            con.close()
    pred_path = ROOT / str(row.get("predictions_path", ""))
    return pd.read_csv(pred_path) if pred_path.exists() else pd.DataFrame()


def load_holdout_with_predictions(row):
    pred = load_predictions_for_row(row)
    if pred.empty or "source_id" not in pred.columns:
        return pd.DataFrame()
    data_path = ROOT / "data/processed/modeling" / f"{row.dataset_variant}_reduced.parquet"
    if not data_path.exists():
        return pd.DataFrame()
    hold_pred = pred[pred["split"].eq("holdout")].copy()
    data = pd.read_parquet(data_path)
    hold = data[data["modeling_split"].eq("holdout")].copy()
    joined = hold.merge(hold_pred[["source_id", "score", "predicted", "threshold"]], on="source_id", how="inner")
    if not WR_CATALOG.empty:
        joined = joined.merge(WR_CATALOG, on="source_id", how="left", suffixes=("", "_catalog"))
    joined["spectral_type"] = joined.get("spectral_type", pd.Series("unknown", index=joined.index)).fillna("unknown")
    joined["wr_family"] = joined.get("wr_family", pd.Series("unknown", index=joined.index)).fillna("unknown")
    joined["rank"] = joined["score"].rank(method="first", ascending=False).astype(int)
    return joined


def add_dynamic_topk(results_df, ks=(10, 50, 100)):
    out = results_df.copy()
    for k in ks:
        out[f"holdout_wr_at_{k}_dynamic"] = np.nan
        out[f"holdout_wr_at_{k}_pct_dynamic"] = np.nan
    for idx, row in out.iterrows():
        joined = load_holdout_with_predictions(row)
        if joined.empty:
            continue
        ordered = joined.sort_values("score", ascending=False)
        wr_total = max(int((joined["target"] == 1).sum()), 1)
        for k in ks:
            recovered = int((ordered.head(k)["target"] == 1).sum())
            out.loc[idx, f"holdout_wr_at_{k}_dynamic"] = recovered
            out.loc[idx, f"holdout_wr_at_{k}_pct_dynamic"] = recovered / wr_total
    for k in ks:
        count = f"holdout_wr_at_{k}"
        pct = f"holdout_wr_at_{k}_pct"
        if count not in out.columns:
            out[count] = out[f"holdout_wr_at_{k}_dynamic"]
        else:
            out[count] = out[count].fillna(out[f"holdout_wr_at_{k}_dynamic"])
        if pct not in out.columns:
            out[pct] = out[f"holdout_wr_at_{k}_pct_dynamic"]
        else:
            out[pct] = out[pct].fillna(out[f"holdout_wr_at_{k}_pct_dynamic"])
    return out


def parse_best_params(value):
    if isinstance(value, dict):
        params = value
    elif pd.isna(value) or value == "":
        params = {}
    else:
        try:
            params = json.loads(value)
        except (TypeError, json.JSONDecodeError):
            params = {}
    return {str(k).replace("estimator__", ""): v for k, v in params.items()}


IMPORTANT_HYPERPARAMS = [
    "n_estimators", "max_depth", "min_samples_leaf", "max_features",
    "learning_rate", "subsample", "colsample_bytree", "reg_lambda", "min_child_weight",
]


def compact_hyperparams(row):
    pieces = []
    if pd.notna(row.get("n_estimators", np.nan)):
        pieces.append(f"trees={int(row['n_estimators'])}")
    if pd.notna(row.get("max_depth", np.nan)):
        pieces.append(f"depth={row['max_depth']}")
    if pd.notna(row.get("learning_rate", np.nan)):
        pieces.append(f"eta={float(row['learning_rate']):.3f}")
    if pd.notna(row.get("min_samples_leaf", np.nan)):
        pieces.append(f"leaf={row['min_samples_leaf']}")
    if pd.notna(row.get("min_child_weight", np.nan)):
        pieces.append(f"child={row['min_child_weight']}")
    if pd.notna(row.get("subsample", np.nan)):
        pieces.append(f"sub={float(row['subsample']):.2f}")
    if pd.notna(row.get("colsample_bytree", np.nan)):
        pieces.append(f"col={float(row['colsample_bytree']):.2f}")
    if pd.notna(row.get("reg_lambda", np.nan)):
        pieces.append(f"lambda={float(row['reg_lambda']):.2f}")
    return " | ".join(pieces) if pieces else "sin params"


def add_hyperparameter_columns(results_df):
    out = results_df.copy()
    parsed = out.get("bayes_best_params", pd.Series([{}] * len(out), index=out.index)).map(parse_best_params)
    numeric_params = [name for name in IMPORTANT_HYPERPARAMS if name != "max_features"]
    for name in IMPORTANT_HYPERPARAMS:
        out[name] = parsed.map(lambda params, n=name: params.get(n, np.nan))
        if name in numeric_params:
            out[name] = pd.to_numeric(out[name], errors="coerce")
    out["hyperparams_compact"] = out.apply(compact_hyperparams, axis=1)
    out["overfit_warning_flag"] = out["selection_status"].eq("overfit_warning")
    return out


results = add_hyperparameter_columns(results)
results = add_dynamic_topk(results, ks=(10, 50, 100))
ranked = rank_models(results, k=100)
top = ranked.head(6).reset_index(drop=True)


In [ ]:
display(top[["model_label", "n_train", "wr_train", "n_holdout", "wr_holdout", "holdout_wr_at_10", "holdout_wr_at_10_pct", "holdout_wr_at_50", "holdout_wr_at_50_pct", "holdout_wr_at_100", "holdout_wr_at_100_pct", "holdout_average_precision", "cv_train_gap_f2", "holdout_cv_gap_f2", "selection_status"]].style.format({"holdout_wr_at_10_pct": "{:.1%}", "holdout_wr_at_50_pct": "{:.1%}", "holdout_wr_at_100_pct": "{:.1%}", "holdout_average_precision": "{:.3f}", "cv_train_gap_f2": "{:.3f}", "holdout_cv_gap_f2": "{:.3f}"}))

## Hyperparameters And Overfit Signals

This view summarizes the complexity selected by `BayesSearchCV`. Overfitting should be read from complexity together with the `train - CV` gap: many trees are not necessarily harmful if the gap remains low, while high depth with a high gap requires caution.

In [ ]:

detail_cols = [
    "model_label", "n_train", "wr_train", "n_holdout", "wr_holdout",
    "holdout_wr_at_10_pct", "holdout_wr_at_50_pct", "holdout_wr_at_100_pct",
    "cv_train_gap_f2", "holdout_cv_gap_f2", "selection_status", "hyperparams_compact",
]
display(top[detail_cols].style.format({
    "holdout_wr_at_10_pct": "{:.1%}",
    "holdout_wr_at_50_pct": "{:.1%}",
    "holdout_wr_at_100_pct": "{:.1%}",
    "cv_train_gap_f2": "{:.3f}",
    "holdout_cv_gap_f2": "{:.3f}",
}))

model_complexity = (
    results.groupby("model", dropna=False)
    .agg(
        configs=("model_label", "size"),
        n_estimators_mean=("n_estimators", "mean"),
        n_estimators_median=("n_estimators", "median"),
        n_estimators_min=("n_estimators", "min"),
        n_estimators_max=("n_estimators", "max"),
        max_depth_mean=("max_depth", "mean"),
        max_depth_median=("max_depth", "median"),
        cv_train_gap_f2_mean=("cv_train_gap_f2", "mean"),
        cv_train_gap_f2_max=("cv_train_gap_f2", "max"),
        holdout_cv_gap_f2_mean=("holdout_cv_gap_f2", "mean"),
        overfit_warning_rate=("overfit_warning_flag", "mean"),
    )
    .sort_values("cv_train_gap_f2_mean", ascending=False)
)
display(model_complexity.style.format({
    "n_estimators_mean": "{:.1f}",
    "n_estimators_median": "{:.0f}",
    "n_estimators_min": "{:.0f}",
    "n_estimators_max": "{:.0f}",
    "max_depth_mean": "{:.1f}",
    "max_depth_median": "{:.0f}",
    "cv_train_gap_f2_mean": "{:.3f}",
    "cv_train_gap_f2_max": "{:.3f}",
    "holdout_cv_gap_f2_mean": "{:.3f}",
    "overfit_warning_rate": "{:.1%}",
}))

plot_df = results.dropna(subset=["n_estimators", "max_depth", "cv_train_gap_f2"]).copy()
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
for model_name, group in plot_df.groupby("model"):
    color = PALETTE.get(model_name, PALETTE["muted"])
    axes[0].scatter(group["n_estimators"], group["cv_train_gap_f2"], s=54, alpha=0.78, color=color, label=model_name)
    axes[1].scatter(group["max_depth"], group["cv_train_gap_f2"], s=54, alpha=0.78, color=color, label=model_name)
for ax in axes:
    ax.axhline(0, color="#111827", lw=1, alpha=0.55)
    ax.axhline(0.15, color="#A8324A", lw=1, ls="--", alpha=0.65)
    ax.set_ylabel("gap F2 train - CV")
axes[0].set_xlabel("number of trees")
axes[1].set_xlabel("maximum depth")
axes[0].set_title("Tree-count complexity")
axes[1].set_title("Depth complexity")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()


## Gaps de estabilidad


In [ ]:
fig, ax = plt.subplots(figsize=(11.2, 5.6))
x = np.arange(len(top)); width = 0.36
ax.bar(x - width/2, top["cv_train_gap_f2"], width=width, color="#B7791F", label="train - CV")
ax.bar(x + width/2, top["holdout_cv_gap_f2"], width=width, color="#2F6F9F", label="holdout - CV")
for i, row in top.iterrows():
    ax.text(i, max(row["cv_train_gap_f2"], row["holdout_cv_gap_f2"], 0) + 0.015, f"{int(row['holdout_wr_at_100'])}/{int(row['wr_holdout'])}", ha="center", fontsize=8)
ax.axhline(0, color="#111827", lw=1)
ax.set_xticks(x, top["model_label"], rotation=35, ha="right", fontsize=8)
ax.set_ylabel("gap F2")
ax.set_title("Overfit gaps annotated with top-100 recovery")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## Train, CV, And Holdout Metrics


In [ ]:
metrics = ["average_precision", "f2_wr", "recall_wr", "precision_wr"]
labels = ["Average precision", "F2", "Recall", "Precision"]
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for ax, metric, label in zip(axes.ravel(), metrics, labels):
    x = np.arange(len(top))
    ax.plot(x, top[f"train_{metric}"], marker="o", label="train", color="#3A9D5D")
    ax.plot(x, top[f"cv_{metric}"], marker="o", label="CV", color="#B7791F")
    ax.plot(x, top[f"holdout_{metric}"], marker="o", label="holdout", color="#2F6F9F")
    ax.set_title(label)
    ax.set_xticks(x, top["model_label"], rotation=35, ha="right", fontsize=8)
axes[0, 0].legend(frameon=False)
fig.suptitle("Metric consistency for selectable models", y=1.02)
plt.tight_layout()
plt.show()


## Confusiones SIMBAD del top auditado

Holdout false positives grouped by SIMBAD type for the audited models.

In [ ]:

rows = []
for _, row in top.iterrows():
    joined = load_holdout_with_predictions(row)
    fp = joined[(joined.get("target", pd.Series(dtype=int)).eq(0)) & (joined.get("predicted", pd.Series(dtype=int)).eq(1))]
    for col in ["simbad_main_type", "simbad_query_type"]:
        if col not in fp.columns:
            continue
        for label, count in fp[col].fillna("missing").value_counts().head(8).items():
            rows.append({"model_label": row["model_label"], "field": col, "type": label, "false_positive_rows": int(count), "total_false_positives": int(len(fp))})
fp_audit = pd.DataFrame(rows)
if fp_audit.empty:
    display(Markdown("No traceable false positives in the audited top models."))
else:
    display(fp_audit)


## Least Recovered WR Subtypes

Subtype from Crowther/GWRC (`Spectral Type`) joined by `source_id`. The model threshold is used to detect families or subtypes with low recovery.

In [ ]:

rows = []
for _, row in top.iterrows():
    joined = load_holdout_with_predictions(row)
    wr = joined[joined.get("target", pd.Series(dtype=int)).eq(1)].copy()
    for col in ["wr_family", "spectral_type"]:
        if col not in wr.columns:
            continue
        g = wr.groupby(col, dropna=False).agg(wr_holdout=("source_id", "size"), wr_recovered=("predicted", "sum"), median_score=("score", "median")).reset_index().rename(columns={col: "wr_type"})
        g["recall_at_threshold"] = g["wr_recovered"] / g["wr_holdout"].replace(0, np.nan)
        g.insert(0, "grouping", col)
        g.insert(0, "model_label", row["model_label"])
        rows.extend(g.to_dict("records"))
wr_type_audit = pd.DataFrame(rows)
if wr_type_audit.empty:
    display(Markdown("Sin WR trazables a subtipo en el top auditado."))
else:
    display(wr_type_audit.sort_values(["grouping", "recall_at_threshold", "wr_holdout"], ascending=[True, True, False]).head(40).style.format({"recall_at_threshold": "{:.1%}", "median_score": "{:.3f}"}))


## Importancia de features


In [ ]:
def load_importance(row):
    p = ROOT / str(row.get("feature_importance_path", ""))
    if p.exists():
        return pd.read_csv(p)
    db = ROOT / "data/databases/training_history.duckdb"
    if db.exists():
        con = duckdb.connect(str(db), read_only=True)
        try:
            return con.execute('''
                SELECT feature, importance_mean, importance_std, importance_type
                FROM feature_importance
                WHERE dataset_variant = ? AND feature_set = ? AND model = ? AND sampler = ?
                ORDER BY importance_mean DESC
            ''', [row.dataset_variant, row.feature_set, row.model, row.sampler]).fetchdf()
        finally:
            con.close()
    return pd.DataFrame()

for _, row in top.head(4).iterrows():
    display(Markdown(f"### {row['model_label']}"))
    imp = load_importance(row)
    if imp.empty:
        display(Markdown("Sin importancia disponible."))
        continue
    display(imp.head(10).style.format(precision=4))
    plot = imp.head(10).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.barh(plot["feature"], plot["importance_mean"], xerr=plot.get("importance_std"), color=PALETTE.get(row["model"], PALETTE["muted"]))
    ax.set_title("Importancia por permutacion")
    ax.set_xlabel("importance mean")
    plt.tight_layout()
    plt.show()
